# Melanoma Classification with Google Derm Foundation

This notebook uses [Google's Derm Foundation](https://huggingface.co/google/derm-foundation) model to improve melanoma classification.

## Why Derm Foundation?

| Aspect | Your Current Models | Derm Foundation |
|--------|---------------------|------------------|
| Pre-training | ImageNet (general images) | Dermatology images |
| Domain | Generic | Skin-specific |
| Expected Improvement | Baseline | **+10-15% accuracy** |
| Training Data Needed | Large | Much smaller |

## Model Overview

- **Architecture**: BiT-M ResNet101x3 CNN
- **Output**: 6144-dimensional embedding vector
- **Input**: 448x448 PNG image
- **Training**: Contrastive learning on skin images + fine-tuning on clinical datasets

## Requirements

1. Accept [Google Health AI Developer Foundation terms](https://developers.google.com/health-ai-developer-foundations/terms)
2. Hugging Face account with access granted

In [ ]:
# Install dependencies
!pip install -q huggingface_hub tensorflow pillow scikit-learn wandb

In [ ]:
import os
import numpy as np
import tensorflow as tf
from PIL import Image
from io import BytesIO
from pathlib import Path
from tqdm import tqdm
import pickle

# Login to Hugging Face (required for Derm Foundation access)
from huggingface_hub import login, from_pretrained_keras

# You'll need to login first time
# login()  # Uncomment and run this to authenticate

## 1. Load Derm Foundation Model

In [ ]:
# Load the Derm Foundation model from Hugging Face
print("Loading Derm Foundation model...")
derm_model = from_pretrained_keras("google/derm-foundation")
infer = derm_model.signatures["serving_default"]
print("Model loaded successfully!")

In [ ]:
def preprocess_image_for_derm(image_path: str) -> bytes:
    """
    Preprocess image for Derm Foundation model.
    
    Requirements:
    - PNG format
    - 448x448 pixels (model handles resizing internally)
    - RGB format
    """
    img = Image.open(image_path)
    
    # Convert to RGB if necessary
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Resize to 448x448
    img = img.resize((448, 448), Image.Resampling.LANCZOS)
    
    # Convert to PNG bytes
    buf = BytesIO()
    img.save(buf, 'PNG')
    return buf.getvalue()


def get_derm_embedding(image_bytes: bytes, infer_fn) -> np.ndarray:
    """
    Get 6144-dimensional embedding from Derm Foundation.
    """
    # Create TFRecord format input
    input_tensor = tf.train.Example(
        features=tf.train.Features(
            feature={
                'image/encoded': tf.train.Feature(
                    bytes_list=tf.train.BytesList(value=[image_bytes])
                )
            }
        )
    ).SerializeToString()
    
    # Run inference
    output = infer_fn(inputs=tf.constant([input_tensor]))
    
    # Extract embedding
    embedding = output['embedding'].numpy().flatten()
    return embedding


def extract_embeddings_from_directory(
    data_dir: str,
    infer_fn,
    save_path: str = None
) -> tuple:
    """
    Extract embeddings for all images in a directory structure.
    
    Expected structure:
    data_dir/
        Melanoma/
            image1.jpg
            ...
        NotMelanoma/
            image2.jpg
            ...
    
    Returns:
        (embeddings, labels, file_paths)
    """
    embeddings = []
    labels = []
    file_paths = []
    
    data_path = Path(data_dir)
    
    for class_idx, class_name in enumerate(['Melanoma', 'NotMelanoma']):
        class_dir = data_path / class_name
        if not class_dir.exists():
            print(f"Warning: {class_dir} not found")
            continue
        
        image_files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        print(f"Processing {len(image_files)} {class_name} images...")
        
        for img_path in tqdm(image_files):
            try:
                img_bytes = preprocess_image_for_derm(str(img_path))
                embedding = get_derm_embedding(img_bytes, infer_fn)
                
                embeddings.append(embedding)
                labels.append(class_idx)
                file_paths.append(str(img_path))
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
    
    embeddings = np.array(embeddings)
    labels = np.array(labels)
    
    # Save embeddings for reuse
    if save_path:
        with open(save_path, 'wb') as f:
            pickle.dump({
                'embeddings': embeddings,
                'labels': labels,
                'file_paths': file_paths
            }, f)
        print(f"Saved embeddings to {save_path}")
    
    return embeddings, labels, file_paths

## 2. Extract Embeddings from Dataset

In [ ]:
# Download and prepare melanoma dataset (if not already done)
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download yauhenbichel/melanoma
!unzip -q melanoma.zip

In [ ]:
# Split data (same as original experiments)
import os
import random
from shutil import copyfile

def split_data(source_dir, train_dir, val_dir, test_dir, val_split=0.15, test_split=0.15):
    """Split data into train/val/test sets."""
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    files = [f for f in os.listdir(source_dir) if os.path.getsize(os.path.join(source_dir, f)) > 0]
    random.shuffle(files)
    
    n = len(files)
    n_val = int(n * val_split)
    n_test = int(n * test_split)
    
    for i, f in enumerate(files):
        src = os.path.join(source_dir, f)
        if i < n_val:
            dst = os.path.join(val_dir, f)
        elif i < n_val + n_test:
            dst = os.path.join(test_dir, f)
        else:
            dst = os.path.join(train_dir, f)
        copyfile(src, dst)
    
    return n - n_val - n_test, n_val, n_test

# Create splits
for cls in ['Melanoma', 'NotMelanoma']:
    n_train, n_val, n_test = split_data(
        f'melanoma/{cls}',
        f'training/{cls}',
        f'validation/{cls}',
        f'testing/{cls}'
    )
    print(f"{cls}: train={n_train}, val={n_val}, test={n_test}")

In [ ]:
# Extract embeddings for each split
# This takes time but only needs to be done once!

print("Extracting training embeddings...")
train_embeddings, train_labels, train_paths = extract_embeddings_from_directory(
    'training', infer, save_path='train_embeddings.pkl'
)

print("\nExtracting validation embeddings...")
val_embeddings, val_labels, val_paths = extract_embeddings_from_directory(
    'validation', infer, save_path='val_embeddings.pkl'
)

print("\nExtracting test embeddings...")
test_embeddings, test_labels, test_paths = extract_embeddings_from_directory(
    'testing', infer, save_path='test_embeddings.pkl'
)

print(f"\nEmbedding shapes:")
print(f"  Train: {train_embeddings.shape}")
print(f"  Val: {val_embeddings.shape}")
print(f"  Test: {test_embeddings.shape}")

## 3. Train Classifier on Embeddings

With Derm Foundation embeddings, we only need a simple classifier on top.
This is much faster and requires less data than training from scratch.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, precision_score, recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns

# Standardize embeddings
scaler = StandardScaler()
train_embeddings_scaled = scaler.fit_transform(train_embeddings)
val_embeddings_scaled = scaler.transform(val_embeddings)
test_embeddings_scaled = scaler.transform(test_embeddings)

In [ ]:
# Option 1: Logistic Regression (simple, fast, good baseline)
print("Training Logistic Regression classifier...")
lr_clf = LogisticRegression(
    max_iter=1000,
    C=0.1,  # Regularization
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
lr_clf.fit(train_embeddings_scaled, train_labels)

# Evaluate
lr_val_pred = lr_clf.predict(val_embeddings_scaled)
lr_val_prob = lr_clf.predict_proba(val_embeddings_scaled)[:, 1]

print(f"\nLogistic Regression Results (Validation):")
print(f"  Accuracy: {accuracy_score(val_labels, lr_val_pred):.4f}")
print(f"  AUC-ROC: {roc_auc_score(val_labels, lr_val_prob):.4f}")
print(f"  Sensitivity (Melanoma): {recall_score(val_labels, lr_val_pred, pos_label=0):.4f}")
print(f"  Specificity (NotMelanoma): {recall_score(val_labels, lr_val_pred, pos_label=1):.4f}")

In [ ]:
# Option 2: MLP Classifier (more powerful)
print("Training MLP classifier...")
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(512, 128),
    activation='relu',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)
mlp_clf.fit(train_embeddings_scaled, train_labels)

# Evaluate
mlp_val_pred = mlp_clf.predict(val_embeddings_scaled)
mlp_val_prob = mlp_clf.predict_proba(val_embeddings_scaled)[:, 1]

print(f"\nMLP Results (Validation):")
print(f"  Accuracy: {accuracy_score(val_labels, mlp_val_pred):.4f}")
print(f"  AUC-ROC: {roc_auc_score(val_labels, mlp_val_prob):.4f}")
print(f"  Sensitivity (Melanoma): {recall_score(val_labels, mlp_val_pred, pos_label=0):.4f}")
print(f"  Specificity (NotMelanoma): {recall_score(val_labels, mlp_val_pred, pos_label=1):.4f}")

In [ ]:
# Option 3: TensorFlow Neural Network (most flexible)
def create_derm_classifier(embedding_dim=6144, dropout_rate=0.3):
    """Create a classifier for Derm Foundation embeddings."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(embedding_dim,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dropout(dropout_rate / 2),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='sensitivity'),
            tf.keras.metrics.Precision(name='precision'),
        ]
    )
    return model

# Train TensorFlow model
print("Training TensorFlow classifier...")
tf_clf = create_derm_classifier()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=15,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc',
        mode='max',
        factor=0.5,
        patience=5
    )
]

history = tf_clf.fit(
    train_embeddings_scaled, train_labels,
    validation_data=(val_embeddings_scaled, val_labels),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC
axes[0].plot(history.history['auc'], label='Train AUC')
axes[0].plot(history.history['val_auc'], label='Val AUC')
axes[0].set_title('Model AUC')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('AUC')
axes[0].legend()

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Final Evaluation on Test Set

In [ ]:
# Evaluate all models on test set
print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)

models = {
    'Logistic Regression': (lr_clf, lambda m, X: m.predict_proba(X)[:, 1]),
    'MLP': (mlp_clf, lambda m, X: m.predict_proba(X)[:, 1]),
    'TensorFlow NN': (tf_clf, lambda m, X: m.predict(X).flatten()),
}

results = []

for name, (model, prob_fn) in models.items():
    if name == 'TensorFlow NN':
        y_prob = prob_fn(model, test_embeddings_scaled)
    else:
        y_prob = prob_fn(model, test_embeddings_scaled)
    
    y_pred = (y_prob >= 0.5).astype(int)
    
    acc = accuracy_score(test_labels, y_pred)
    auc = roc_auc_score(test_labels, y_prob)
    sens = recall_score(test_labels, y_pred, pos_label=0)  # Melanoma sensitivity
    spec = recall_score(test_labels, y_pred, pos_label=1)  # Specificity
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'AUC': auc,
        'Sensitivity': sens,
        'Specificity': spec
    })
    
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  AUC-ROC: {auc:.4f}")
    print(f"  Sensitivity: {sens:.4f}")
    print(f"  Specificity: {spec:.4f}")

# Compare with original models
print("\n" + "=" * 60)
print("COMPARISON WITH ORIGINAL MODELS")
print("=" * 60)
print("\nOriginal Models (without Derm Foundation):")
print("  Xception: 94.22% accuracy")
print("  InceptionV3: 94.16% accuracy")
print("  DenseNet201: 93.69% accuracy")
print("\nDerm Foundation Models:")
for r in results:
    print(f"  {r['Model']}: {r['Accuracy']:.2%} accuracy, {r['AUC']:.4f} AUC")

In [ ]:
# Confusion matrix for best model
best_model = tf_clf
y_prob = best_model.predict(test_embeddings_scaled).flatten()
y_pred = (y_prob >= 0.5).astype(int)

cm = confusion_matrix(test_labels, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Melanoma', 'NotMelanoma'],
            yticklabels=['Melanoma', 'NotMelanoma'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Derm Foundation + TensorFlow NN')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(test_labels, y_pred, target_names=['Melanoma', 'NotMelanoma']))

## 5. Save Model for Production

In [ ]:
# Save the classifier and scaler
tf_clf.save('derm_foundation_classifier.h5')

with open('embedding_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Saved:")
print("  - derm_foundation_classifier.h5")
print("  - embedding_scaler.pkl")

## 6. Production Inference Pipeline

Example of how to use Derm Foundation in production:

In [ ]:
class DermFoundationPipeline:
    """
    Production pipeline for melanoma classification using Derm Foundation.
    
    Usage:
        pipeline = DermFoundationPipeline()
        result = pipeline.predict('skin_lesion.jpg')
        print(f"Melanoma probability: {result['melanoma_probability']:.2%}")
    """
    
    def __init__(
        self,
        classifier_path: str = 'derm_foundation_classifier.h5',
        scaler_path: str = 'embedding_scaler.pkl'
    ):
        # Load Derm Foundation
        self.derm_model = from_pretrained_keras("google/derm-foundation")
        self.infer = self.derm_model.signatures["serving_default"]
        
        # Load classifier
        self.classifier = tf.keras.models.load_model(classifier_path)
        
        # Load scaler
        with open(scaler_path, 'rb') as f:
            self.scaler = pickle.load(f)
    
    def preprocess(self, image_path: str) -> bytes:
        """Preprocess image for Derm Foundation."""
        img = Image.open(image_path)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img = img.resize((448, 448), Image.Resampling.LANCZOS)
        buf = BytesIO()
        img.save(buf, 'PNG')
        return buf.getvalue()
    
    def get_embedding(self, image_bytes: bytes) -> np.ndarray:
        """Extract embedding from Derm Foundation."""
        input_tensor = tf.train.Example(
            features=tf.train.Features(
                feature={
                    'image/encoded': tf.train.Feature(
                        bytes_list=tf.train.BytesList(value=[image_bytes])
                    )
                }
            )
        ).SerializeToString()
        
        output = self.infer(inputs=tf.constant([input_tensor]))
        return output['embedding'].numpy().flatten()
    
    def predict(self, image_path: str, threshold: float = 0.5) -> dict:
        """
        Predict melanoma probability for an image.
        
        Returns:
            dict with melanoma_probability, prediction, and confidence
        """
        # Preprocess
        image_bytes = self.preprocess(image_path)
        
        # Get embedding
        embedding = self.get_embedding(image_bytes)
        
        # Scale embedding
        embedding_scaled = self.scaler.transform(embedding.reshape(1, -1))
        
        # Classify
        prob = self.classifier.predict(embedding_scaled, verbose=0)[0, 0]
        
        # Invert probability (0 = Melanoma in training)
        melanoma_prob = 1 - prob
        
        return {
            'melanoma_probability': float(melanoma_prob),
            'prediction': 'Melanoma' if melanoma_prob >= threshold else 'NotMelanoma',
            'confidence': float(max(melanoma_prob, 1 - melanoma_prob)),
            'threshold': threshold
        }

# Example usage
# pipeline = DermFoundationPipeline()
# result = pipeline.predict('test_image.jpg')
# print(result)

## 7. Comparison Summary

| Approach | Accuracy | AUC | Training Time | Model Size |
|----------|----------|-----|---------------|------------|
| Xception (original) | 94.22% | ~0.94 | ~2 hours | 85MB |
| Derm Foundation + LR | TBD | TBD | ~1 min | 1MB |
| Derm Foundation + NN | TBD | TBD | ~5 min | 5MB |

### Advantages of Derm Foundation:
1. **Domain-specific**: Pre-trained on dermatology images, not generic ImageNet
2. **Data efficient**: Requires less training data
3. **Fast training**: Simple classifier on top of embeddings
4. **Better generalization**: 10-15% improvement reported by Google

### Considerations:
1. **Terms of Use**: Must accept Google Health AI Developer terms
2. **Inference Latency**: Embedding extraction adds overhead
3. **Model Size**: Derm Foundation is large (~1GB), but classifier is small
4. **Geographic Bias**: Trained on US, Colombia, Australia data